In [11]:
import sympy as sp
import re

In [12]:
# symbols

lam_x, lam_y, theta_z = sp.symbols('lambda_x lambda_y theta_z')
pi = sp.pi

y_vec = sp.Matrix([sp.Symbol(f"rotated[{i}]") for i in range(15)])

In [13]:
q = sp.Matrix([
    (2/sp.Integer(5)) * sp.sqrt(pi),
    0,
    0,
    (8/sp.Integer(7)) * sp.sqrt(pi/5),
    0,
    0,
    0,
    0,
    0,
    0,
    (16/sp.Integer(105)) * sp.sqrt(pi),
    0,
    0,
    0,
    0
])

# B_z (15x5 matrix)
B = sp.Matrix([
    [(2/sp.Integer(5))*sp.sqrt(pi), 0, 0, 0, 0],
    [0, (4/sp.Integer(7))*sp.sqrt(3*pi/5), 0, 0, 0],
    [0, 0, 0, 0, 0],
    [-(4/sp.Integer(7))*sp.sqrt(pi/5), 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, (4/sp.Integer(7))*sp.sqrt(3*pi/5), 0, 0],
    [0, 0, 0, (2/sp.Integer(3))*sp.sqrt(pi/35), 0],
    [0, 0, 0, 0, 0],
    [0, -(4/sp.Integer(21))*sp.sqrt(pi/5), 0, 0, 0],
    [0, 0, 0, 0, 0],
    [(2/sp.Integer(35))*sp.sqrt(pi), 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, -(4/sp.Integer(21))*sp.sqrt(pi/5), 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, (2/sp.Integer(3))*sp.sqrt(pi/35)]
])

# s_z (5x1 vector)
s = sp.Matrix([
    lam_x + lam_y,
    (lam_x - lam_y) * sp.sin(2*theta_z),
    (lam_x - lam_y) * sp.cos(2*theta_z),
    (lam_x + lam_y) * sp.sin(4*theta_z),
    (lam_x + lam_y) * sp.cos(4*theta_z)
])

In [14]:
B.T * B

Matrix([
[8*pi/35,         0,         0,        0,        0],
[      0, 64*pi/315,         0,        0,        0],
[      0,         0, 64*pi/315,        0,        0],
[      0,         0,         0, 4*pi/315,        0],
[      0,         0,         0,        0, 4*pi/315]])

In [15]:
# solution via least squares

s_star = (B.T * B).inv() * B.T * (y_vec - q)
s_star = sp.simplify(s_star)

#print(sp.latex(s_star))
print(to_ccode_vector(s_star))

{
    constant<T>(0.98733177120857341)*rotated[0] + constant<T>(0.14104739588693904)*rotated[10] - constant<T>(0.63078313050503998)*rotated[3] - constant<T>(0.16666666666666663),
    constant<T>(1.2291169844160892)*rotated[1] - constant<T>(0.23654367393939002)*rotated[8],
    constant<T>(-0.23654367393939002)*rotated[12] + constant<T>(1.2291169844160892)*rotated[5],
    constant<T>(5.0066858835934092)*rotated[6],
    constant<T>(5.0066858835934092)*rotated[14]
};


In [16]:
proj = q + B * s_star
proj = sp.simplify(proj)

# print(sp.latex(proj))
proj

Matrix([
[             7*rotated[0]/10 + rotated[10]/10 - sqrt(5)*rotated[3]/5 + sqrt(pi)/3],
[                                       27*rotated[1]/28 - 3*sqrt(3)*rotated[8]/28],
[                                                                                0],
[sqrt(5)*(-21*rotated[0] - 3*rotated[10] + 6*sqrt(5)*rotated[3] + 26*sqrt(pi))/105],
[                                                                                0],
[                                     -3*sqrt(3)*rotated[12]/28 + 27*rotated[5]/28],
[                                                                       rotated[6]],
[                                                                                0],
[                                         -3*sqrt(3)*rotated[1]/28 + rotated[8]/28],
[                                                                                0],
[              rotated[0]/10 + rotated[10]/70 - sqrt(5)*rotated[3]/35 + sqrt(pi)/7],
[                                                       

In [17]:
out = ""

out += "vec15(\n"

for i in range(15):
    out += "    " + sp.ccode(proj[i]) + ",\n"
out = out.removesuffix(",\n")
out += "\n);"

float_re = re.compile(
    r'(?<![\w.])'
    r'([+-]?(?:\d+\.\d*|\.\d+)(?:[eE][+-]?\d+)?'
    r'|[+-]?\d+[eE][+-]?\d+)'
)

def wrap_constants(code):
    return float_re.sub(r'constant<T>(\1)', code)

def to_ccode_vector(expr):
    ev = expr.evalf()
    out = "{\n"

    for i in range(len(ev)):
        code = sp.ccode(sp.simplify(ev[i]))
        code = wrap_constants(code)
        out += f"    {code}"
        if i != len(ev) - 1:
            out += ",\n"

    out += "\n};"
    return out

print(to_ccode_vector(proj))

{
    constant<T>(0.69999999999999996)*rotated[0] + constant<T>(0.10000000000000001)*rotated[10] - constant<T>(0.44721359549995798)*rotated[3] + constant<T>(0.59081795030183859),
    constant<T>(0.9642857142857143)*rotated[1] - constant<T>(0.18557687223952254)*rotated[8],
    0,
    constant<T>(-0.44721359549995798)*rotated[0] - constant<T>(0.063887656499993992)*rotated[10] + constant<T>(0.28571428571428575)*rotated[3] + constant<T>(0.98139533083577413),
    0,
    constant<T>(-0.18557687223952254)*rotated[12] + constant<T>(0.9642857142857143)*rotated[5],
    rotated[6],
    0,
    constant<T>(-0.18557687223952254)*rotated[1] + constant<T>(0.035714285714285712)*rotated[8],
    0,
    constant<T>(0.10000000000000001)*rotated[0] + constant<T>(0.014285714285714285)*rotated[10] - constant<T>(0.063887656499993992)*rotated[3] + constant<T>(0.25320769298650225),
    0,
    constant<T>(0.035714285714285712)*rotated[12] - constant<T>(0.18557687223952254)*rotated[5],
    0,
    rotated[14]
};


In [18]:
numeric_proj = proj.evalf()

out = ""

out += "vec15(\n"

for i in range(15):
    out += "    " + sp.ccode(numeric_proj[i]) + ",\n"
out = out.removesuffix(",\n")
out += "\n);"

print(out)

vec15(
    0.69999999999999996*rotated[0] + 0.10000000000000001*rotated[10] - 0.44721359549995798*rotated[3] + 0.59081795030183859,
    0.9642857142857143*rotated[1] - 0.18557687223952254*rotated[8],
    0,
    -0.44721359549995798*rotated[0] - 0.063887656499993992*rotated[10] + 0.28571428571428575*rotated[3] + 0.98139533083577413,
    0,
    -0.18557687223952254*rotated[12] + 0.9642857142857143*rotated[5],
    rotated[6],
    0,
    -0.18557687223952254*rotated[1] + 0.035714285714285712*rotated[8],
    0,
    0.10000000000000001*rotated[0] + 0.014285714285714285*rotated[10] - 0.063887656499993992*rotated[3] + 0.25320769298650225,
    0,
    0.035714285714285712*rotated[12] - 0.18557687223952254*rotated[5],
    0,
    rotated[14]
);
